# Clase 5 · La gramática de la duda

**Estadística Descriptiva e Inferencial** · Módulo 2 · Sesión 5 de 14

Pruebas de hipótesis, el p-valor, los errores tipo I y II, la potencia y el problema
de hacer muchas pruebas a la vez.

---

### Lo que este notebook hace distinto

Calcular un p-valor es una línea de código y lo terminarás en el bloque 1. El resto del
laboratorio trata de algo más difícil: **comprobar por simulación que las promesas del
método se cumplen — o no.**

En particular, en el bloque 5 vas a hacer p-hacking **sobre datos sin ningún efecto real**
y vas a encontrar un resultado «significativo». Esa experiencia vale más que cualquier
advertencia.

| Bloque | Tema | Min |
|---|---|---|
| 1 | Tu primera prueba, a mano y con scipy | 12 |
| 2 | La dualidad IC ↔ prueba | 13 |
| 3 | Tipo I, tipo II y potencia por simulación | 15 |
| 4 | Significativo vs importante | 13 |
| 5 | P-hacking en vivo y corrección | 12 |

> **SEED = 42.** El bloque 5 es el que hay que hacer aunque el tiempo se acabe.

## Celda 0 · Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm, t

SEED = 42
rng = np.random.default_rng(SEED)

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (9, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-6):
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada (obtenido = None)")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}: obtenido = {float(obtenido):.6f} | "
          f"esperado = {float(esperado):.6f} (tolerancia {tol})")
    if not ok:
        print("      -> revisa este paso antes de continuar.")
    return ok

def check_bool(nombre, condicion, pista=""):
    print(f"{'[OK]' if condicion else '[X ]'} {nombre}")
    if not condicion and pista:
        print(f"      -> {pista}")
    return bool(condicion)

# ── La muestra que recorre toda la clase (la misma de la Clase 4) ─────────
XB, S, NM = 3125, 2210, 40
DF = NM - 1
EE = S / np.sqrt(NM)

print("Entorno listo | SEED =", SEED)
print(f"Muestra: n = {NM}, x_barra = S/ {XB}, s = S/ {S}, EE = {EE:.2f}")

### Tabla de símbolos → código

| Símbolo | Qué es | En el código |
|---|---|---|
| H₀ | hipótesis nula | el valor `mu0` que se pone a prueba |
| α | tolerancia al error tipo I | `alpha = 0.05` |
| t | estadístico de prueba | `t_obs` |
| p | p-valor | `p_val` |
| ν, gl | grados de libertad | `df = n - 1` |
| 1 − β | potencia | `potencia` |
| d | tamaño del efecto (Cohen) | `d` |

**La función que resume todo el bloque 1:** `stats.ttest_1samp(muestra, popmean)`.

---
# Bloque 1 · Tu primera prueba, a mano y con scipy  ·  12 min

Los cinco pasos de la slide 6, en código:

1. H₀: μ = μ₀ y H₁: μ ≠ μ₀
2. α = 0.05
3. t = (x̄ − μ₀) / (s/√n)
4. p = 2 · P(T > |t|)
5. p < α → se rechaza

Primero con aritmética, para entender de dónde sale cada número. Después con `scipy`,
para no volver a hacerlo nunca a mano.

### Ejercicio 1.1 — El estadístico y el p-valor, a mano

Usa la muestra del enunciado y prueba H₀: μ = 2 300.

**Pista:** la cola derecha de la t es `t.sf(x, df)` («survival function»). Para el
p-valor bilateral se multiplica por 2.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
mu0 = 2300

t_obs = (XB - mu0) / EE
p_val = 2 * t.sf(abs(t_obs), DF)

print(f"Paso 1  H0: mu = {mu0}   H1: mu != {mu0}")
print(f"Paso 2  alpha = 0.05")
print(f"Paso 3  t = ({XB} - {mu0}) / {EE:.2f} = {t_obs:.4f}")
print(f"Paso 4  cola derecha = {t.sf(abs(t_obs), DF):.6f}   ->  p bilateral = {p_val:.6f}")
print(f"Paso 5  p = {p_val:.4f} < 0.05  ->  se RECHAZA H0")
print()
print("Redaccion completa, como en la slide 20:")
print(f"  'En una muestra de {NM} transacciones el monto promedio fue S/ {XB},")
print(f"   frente al valor de referencia de S/ {mu0}. La diferencia es de S/ {XB-mu0}")
print(f"   (t = {t_obs:.2f}, gl = {DF}, p = {p_val:.3f}).'")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("estadístico t", t_obs, 2.361024, tol=1e-4),
     check("p-valor bilateral", p_val, 0.023320, tol=1e-5),
     check_bool("y la decisión es rechazar H₀", p_val < 0.05)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — Lo mismo con scipy, y la comprobación de que coincide

`stats.ttest_1samp` necesita la muestra completa, no solo el resumen. Vamos a construir
una muestra artificial que tenga **exactamente** la media y la desviación del enunciado,
para poder comparar los dos caminos.

**Pista:** si tomas cualquier muestra, la centras y la escalas, puedes forzar su media y
su `s`: `z = (x - x.mean()) / x.std(ddof=1)` y después `XB + S * z`.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
g = np.random.default_rng(SEED)
bruta = g.normal(0, 1, NM)

z = (bruta - bruta.mean()) / bruta.std(ddof=1)
muestra = XB + S * z

print(f"media de la muestra construida : {muestra.mean():.6f}   (queriamos {XB})")
print(f"desviacion (ddof=1)            : {muestra.std(ddof=1):.6f}   (queriamos {S})")
print()

res = stats.ttest_1samp(muestra, mu0)
print(f"scipy   -> t = {res.statistic:.6f}   p = {res.pvalue:.6f}")
print(f"a mano  -> t = {t_obs:.6f}   p = {p_val:.6f}")
print()
print("Coinciden en los seis decimales. Ya no hay que volver a hacerlo a mano —")
print("pero ahora sabes exactamente que hace esa funcion por dentro.")
print()
# El IC que scipy devuelve es el MISMO de la Clase 4
ic = res.confidence_interval(confidence_level=0.95)
print(f"Y el IC al 95 % que da scipy: [{ic.low:,.1f}, {ic.high:,.1f}]")
print("Es el intervalo de la Clase 4. Esa coincidencia es el bloque 2.")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("media de la muestra construida", muestra.mean(), XB, tol=1e-6),
     check("desviación de la muestra construida", muestra.std(ddof=1), S, tol=1e-6),
     check("t de scipy vs t a mano", res.statistic, t_obs, tol=1e-9),
     check("p de scipy vs p a mano", res.pvalue, p_val, tol=1e-12)]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.2")

---
# Bloque 2 · La dualidad IC ↔ prueba  ·  13 min

La slide 8 afirmó algo fuerte: **un intervalo de confianza es exactamente el conjunto de
valores que una prueba no rechazaría.** Vamos a comprobarlo en lugar de creerlo.

El plan: recorrer cientos de valores de μ₀, correr una prueba para cada uno, quedarnos
con los que NO se rechazan, y ver si el mínimo y el máximo de ese conjunto son los
límites del intervalo.

### Ejercicio 2.1 — Recorre los μ₀ y quédate con los no rechazados

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
candidatos = np.arange(2000, 4400, 0.5)

t_todos    = (XB - candidatos) / EE            # vectorizado: un t por candidato
p_valores  = 2 * t.sf(np.abs(t_todos), DF)
no_rechazados = candidatos[p_valores >= 0.05]

print(f"{len(no_rechazados)} de {len(candidatos)} valores NO se rechazan")
print(f"El conjunto de no rechazados va de {no_rechazados.min():,.1f} a {no_rechazados.max():,.1f}")
print()

# Y ahora el intervalo de confianza, calculado como en la Clase 4
critico = t.ppf(0.975, DF)
ic_lo, ic_hi = XB - critico * EE, XB + critico * EE
print(f"El intervalo de confianza al 95 %:     [{ic_lo:,.1f}, {ic_hi:,.1f}]")
print(f"El conjunto de mu0 no rechazados:      [{no_rechazados.min():,.1f}, {no_rechazados.max():,.1f}]")
print()
print("Son el mismo conjunto (salvo el paso de 0.5 de la rejilla).")
print("No es una coincidencia: es la misma desigualdad despejada de dos maneras.")

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
critico = t.ppf(0.975, DF)
ic_lo, ic_hi = XB - critico * EE, XB + critico * EE
r = [check("límite inferior del IC", ic_lo, 2418.2, tol=0.5),
     check("límite superior del IC", ic_hi, 3831.8, tol=0.5),
     check_bool("el menor μ₀ no rechazado coincide con el límite inferior (±1)",
                abs(no_rechazados.min() - ic_lo) <= 1.0,
                "revisa que uses p >= 0.05, no p > 0.05"),
     check_bool("el mayor μ₀ no rechazado coincide con el límite superior (±1)",
                abs(no_rechazados.max() - ic_hi) <= 1.0)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

In [ ]:
# ── DEMOSTRACIÓN: la slide 8, dibujada con tus datos ─────────────────────
fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(candidatos, p_valores, color=BLUE, lw=2.5, label="p-valor de cada μ₀")
ax.axhline(0.05, color=MAG, ls="--", lw=2, label="α = 0.05")
ax.axvspan(ic_lo, ic_hi, color=BLUE, alpha=0.10,
           label="intervalo de confianza al 95 %")
ax.axvline(XB, color=NAVY, lw=1.5, ls=":", label=f"x̄ = {XB}")

for mu, col, txt in [(2300, MAG, "se rechaza"), (2800, GREEN, "no se rechaza")]:
    pv = 2 * t.sf(abs((XB - mu) / EE), DF)
    ax.plot([mu], [pv], "o", color=col, ms=9, zorder=5)
    ax.annotate(f"μ₀={mu}\np={pv:.3f}\n{txt}", (mu, pv),
                textcoords="offset points", xytext=(5, 25),
                fontsize=9, color=col, fontweight="bold")

ax.set_xlabel("valor de μ₀ que se pone a prueba")
ax.set_ylabel("p-valor")
ax.set_title("El p-valor de cada hipótesis, y el intervalo de confianza",
             color=NAVY, fontweight="bold")
ax.legend(frameon=False, fontsize=9, loc="upper right")
plt.tight_layout(); plt.show()

print("Lectura del grafico: la curva azul cruza la linea de alpha exactamente en los")
print("bordes de la banda sombreada. Los mu0 con p por encima de 0.05 son los del")
print("intervalo; los de fuera se rechazan. Una sola figura, las dos herramientas.")

### Ejercicio 2.2 — La consecuencia práctica

Si el intervalo contiene el resultado de todas las pruebas posibles, entonces reportar
el intervalo es **estrictamente más informativo** que reportar un p-valor.

Compruébalo: ¿qué valores de referencia rechazarías y cuáles no, leyendo solo el intervalo?

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
referencias = [2200, 2418, 2500, 2800, 3125, 3500, 3832, 4000]

filas = []
for ref in referencias:
    dentro_ic = bool(ic_lo <= ref <= ic_hi)
    p_ref     = 2 * t.sf(abs((XB - ref) / EE), DF)
    rechaza   = bool(p_ref < 0.05)
    filas.append({"referencia": ref, "dentro_IC": dentro_ic,
                  "p": round(p_ref, 4), "rechaza": rechaza,
                  "coinciden": dentro_ic != rechaza})

tabla_dual = pd.DataFrame(filas)
print(tabla_dual.to_string(index=False))
print()
print("La columna 'coinciden' es True en todas las filas: 'dentro del IC' y 'no se")
print("rechaza' son siempre la misma respuesta.")
print()
print("Fijate en las filas de 2418 y 3832: estan justo en el borde y su p vale casi")
print("exactamente 0.05. Ahi se ve que el limite del intervalo ES el umbral de decision.")

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
tabla_dual = pd.DataFrame(filas)
r = [check_bool("las dos decisiones coinciden en las 8 referencias",
                tabla_dual["coinciden"].all(),
                "'dentro_IC' debe ser el opuesto de 'rechaza' en cada fila"),
     check_bool("el p-valor en el borde del IC vale ≈ 0.05",
                abs(tabla_dual.loc[tabla_dual["referencia"] == 2418, "p"].iloc[0] - 0.05) < 0.002)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.2")

---
# Bloque 3 · Tipo I, tipo II y potencia, medidos  ·  15 min

α y β no son abstracciones: son tasas que se pueden contar. Vamos a contarlas.

- **Tipo I:** simulamos un mundo donde H₀ **es verdadera** y contamos cuántas veces la
  rechazamos. Debería salir 5 %.
- **Potencia:** simulamos un mundo donde H₀ **es falsa** y contamos cuántas veces la
  detectamos. Ahí es donde aparecen las sorpresas.

### Ejercicio 3.1 — La tasa real de error tipo I

Genera 20 000 muestras de una población donde H₀ es **exactamente verdadera** (μ = μ₀)
y cuenta la proporción de rechazos.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
MU_VERDADERO, SD_POB, n_, reps = 3000, 2200, 40, 20_000
g = np.random.default_rng(SEED)

muestras = g.normal(MU_VERDADERO, SD_POB, size=(reps, n_))

x_barras = muestras.mean(axis=1)
eses     = muestras.std(axis=1, ddof=1)
t_stats  = (x_barras - MU_VERDADERO) / (eses / np.sqrt(n_))
p_vals   = 2 * t.sf(np.abs(t_stats), n_ - 1)

tasa_tipo_I = 100 * np.mean(p_vals < 0.05)

print(f"H0 era VERDADERA en las {reps:,} simulaciones.")
print(f"La rechazamos en {tasa_tipo_I:.2f} % de los casos.")
print(f"alpha que habiamos fijado: 5.00 %")
print()
print("El metodo cumple exactamente lo que promete: si H0 es verdadera, la rechazas")
print("el 5 % de las veces. Esos rechazos NO son errores de calculo: son el precio de alpha.")
print()
# Y la distribucion de p-valores bajo H0 es uniforme: un resultado bonito y util
plt.hist(p_vals, bins=40, color=BLUE, alpha=0.8, edgecolor="white", linewidth=0.4)
plt.axhline(reps / 40, color=MAG, ls="--", lw=2, label="lo que espera una uniforme")
plt.xlabel("p-valor"); plt.ylabel("frecuencia")
plt.title("Distribución de los p-valores cuando H₀ es verdadera",
          color=NAVY, fontweight="bold")
plt.legend(frameon=False); plt.show()

print("Dato que vale recordar: si H0 es verdadera, el p-valor se distribuye UNIFORME")
print("entre 0 y 1. Es igual de probable obtener p=0.03 que p=0.83. Por eso basta con")
print("repetir el analisis muchas veces para encontrar un p pequeno sin ningun efecto real.")

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
r = [check_bool("la tasa de error tipo I está entre 4.5 % y 5.5 %",
                4.5 <= tasa_tipo_I <= 5.5,
                f"te dio {tasa_tipo_I:.2f} %; revisa que centres el t en MU_VERDADERO")]
print()
print("3.1 OK" if all(r) else "Revisa 3.1")

### Ejercicio 3.2 — La potencia, por simulación

Ahora H₀ es **falsa**: la población tiene μ = 3 000 pero H₀ afirma 2 800. La diferencia
real es de 200 soles, es decir un efecto de d = 200/2 200 ≈ 0.09.

Mide qué porcentaje de las veces la detectas, para n creciente.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
MU_REAL_POB, MU0_FALSO = 3000, 2800
d_real = (MU_REAL_POB - MU0_FALSO) / SD_POB

filas = []
for n_ in [40, 100, 400, 1000, 2000]:
    g = np.random.default_rng(SEED)
    m  = g.normal(MU_REAL_POB, SD_POB, size=(5000, n_))
    xb = m.mean(axis=1)
    ss = m.std(axis=1, ddof=1)
    ts = (xb - MU0_FALSO) / (ss / np.sqrt(n_))
    pv = 2 * t.sf(np.abs(ts), n_ - 1)
    filas.append({"n": n_, "potencia_%": 100 * np.mean(pv < 0.05)})

tabla_pot = pd.DataFrame(filas)
print(f"Efecto real: {MU_REAL_POB - MU0_FALSO} soles  ->  d = {d_real:.3f} (pequeno)")
print()
print(tabla_pot.round(1).to_string(index=False))
print()
print("Con n = 40 la potencia es de apenas un dígito o dos: el estudio esta")
print("practicamente condenado a NO detectar una diferencia de 200 soles que SI existe.")
print("Y si no la detecta, alguien escribira 'no hay diferencia significativa'.")
print()
print("Para llegar al 80 % de potencia con este efecto hacen falta unos cuantos miles")
print("de observaciones. Eso es lo que hay que saber ANTES de disenar el estudio.")

plt.plot(tabla_pot["n"], tabla_pot["potencia_%"], "o-", color=MAG, lw=2.5)
plt.axhline(80, color=NAVY, ls="--", lw=1.5, label="80 % (estándar profesional)")
plt.xscale("log"); plt.xlabel("tamaño de muestra (escala log)")
plt.ylabel("potencia (%)"); plt.ylim(0, 100)
plt.title(f"Potencia para detectar un efecto de d = {d_real:.2f}",
          color=NAVY, fontweight="bold")
plt.legend(frameon=False); plt.show()

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
tabla_pot = pd.DataFrame(filas)
r = [check_bool("la potencia crece con n", tabla_pot["potencia_%"].is_monotonic_increasing),
     check_bool("con n = 40 la potencia es baja (< 25 %)",
                tabla_pot.loc[0, "potencia_%"] < 25,
                "con un efecto de d≈0.09 y n=40, la potencia debe ser muy baja"),
     check_bool("con n = 2000 ya supera el 80 %",
                tabla_pot.iloc[-1]["potencia_%"] > 80)]
print()
print("Y compara con la fórmula exacta de statsmodels:")
from statsmodels.stats.power import TTestPower
for n_ in [40, 400, 2000]:
    ex_ = TTestPower().power(effect_size=(3000 - 2800) / 2200, nobs=n_,
                            alpha=0.05, alternative="two-sided")
    sim = tabla_pot.loc[tabla_pot["n"] == n_, "potencia_%"].iloc[0]
    print(f"  n={n_:>5}: simulada {sim:5.1f} %   exacta {100*ex_:5.1f} %")
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa 3.2")

---
# Bloque 4 · Significativo no es importante  ·  13 min

La slide 14 mostró una tabla. Ahora la construyes tú y ves el mecanismo desde dentro:
**el mismo efecto, sin cambiar nada, se vuelve «significativo» solo por tener más datos.**

### Ejercicio 4.1 — El mismo efecto con n creciente

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
d = 0.10

filas = []
for n_ in [30, 100, 400, 1600, 6400, 25600]:
    t_teorico = d * np.sqrt(n_)
    p         = 2 * t.sf(t_teorico, n_ - 1)
    filas.append({"n": n_, "d": d, "t": round(t_teorico, 3),
                  "p": p, "significativo": p < 0.05})

tabla_ef = pd.DataFrame(filas)
tabla_ef["p_texto"] = tabla_ef["p"].apply(
    lambda v: "< 0.00001" if v < 1e-5 else f"{v:.5f}")
print(tabla_ef[["n", "d", "t", "p_texto", "significativo"]].to_string(index=False))
print()
print("El efecto es 0.10 en las seis filas. No cambio nunca.")
print("Lo unico que cambio fue n, y el p-valor se desplomo de 0.59 a menos de 0.00001.")
print()
print("Conclusion operativa para quien trabaja con 40 000 filas:")
print("  la significancia estadistica deja de ser informativa en ese regimen.")
print("  La pregunta util es 'cuanto', no 'es significativo'.")

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
tabla_ef = pd.DataFrame(filas)
r = [check("p con n = 30", tabla_ef.loc[0, "p"], 0.588074, tol=1e-4),
     check("p con n = 400", tabla_ef.loc[2, "p"], 0.046178, tol=1e-4),
     check_bool("el efecto d es el mismo en todas las filas",
                tabla_ef["d"].nunique() == 1),
     check_bool("con n = 30 no es significativo y con n = 400 sí",
                (not tabla_ef.loc[0, "significativo"]) and tabla_ef.loc[2, "significativo"])]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

### Ejercicio 4.2 — Lo que sí hay que reportar

Toma la muestra del bloque 1 y produce el reporte completo de la slide 20: efecto en
unidades del negocio, d de Cohen, intervalo de confianza, p-valor exacto y grados de
libertad.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
mu0 = 2300

efecto_soles = XB - mu0
d_cohen      = efecto_soles / S
critico      = t.ppf(0.975, DF)
ic_lo_r      = XB - critico * EE
ic_hi_r      = XB + critico * EE

print("REPORTE COMPLETO")
print("-" * 62)
print(f"  Tamano de muestra        : n = {NM}")
print(f"  Promedio observado       : S/ {XB:,}")
print(f"  Valor de referencia (H0) : S/ {mu0:,}")
print(f"  Efecto                   : S/ {efecto_soles:,}  (d de Cohen = {d_cohen:.3f})")
print(f"  IC 95 % de la media      : [S/ {ic_lo_r:,.0f}, S/ {ic_hi_r:,.0f}]")
print(f"  Prueba                   : t de una muestra, bilateral")
print(f"  Estadistico              : t = {t_obs:.3f}, gl = {DF}")
print(f"  p-valor                  : p = {p_val:.4f}")
print(f"  Pruebas realizadas       : 1, definida antes del analisis")
print("-" * 62)
print()
print(f"El efecto es de S/ {efecto_soles} y d = {d_cohen:.2f}, que es un efecto GRANDE.")
print("Aqui el resultado es significativo Y importante. No siempre coincide,")
print("y por eso hay que reportar las dos cosas.")

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check("efecto en soles", efecto_soles, 825, tol=0.5),
     check("d de Cohen", d_cohen, 0.373303, tol=1e-4),
     check("límite inferior del IC", ic_lo_r, 2418.2, tol=0.5),
     check_bool("el IC NO contiene a μ₀ = 2300 (coherente con rechazar)",
                not (ic_lo_r <= 2300 <= ic_hi_r))]
print()
print("Bloque 4 COMPLETO" if all(r) else "Revisa 4.2")

---
# Bloque 5 · P-hacking en vivo  ·  12 min

**Este es el bloque que hay que hacer aunque el tiempo se acabe.**

Vamos a generar datos donde **no existe ningún efecto**: la variable de interés es ruido
puro, sin relación con nada. Después vamos a buscar en subgrupos hasta encontrar algo
«significativo». Lo vamos a encontrar.

No hace falta mala intención. Solo curiosidad sin registro previo.

In [ ]:
# ── DEMOSTRACIÓN: datos SIN ningún efecto real ───────────────────────────
g = np.random.default_rng(SEED)
N = 2000

datos = pd.DataFrame({
    # La variable de interes: ruido puro, misma distribucion para todos.
    "monto":    g.normal(3000, 800, N),
    # Y un montón de variables de segmentación, todas independientes del monto.
    "region":   g.choice(["norte", "centro", "sur", "oriente"], N),
    "segmento": g.choice(["retail", "pyme", "corporativo"], N),
    "canal":    g.choice(["app", "web", "agencia"], N),
    "antiguo":  g.choice([True, False], N),
    "digital":  g.choice([True, False], N),
})

print(f"{N} filas. La columna 'monto' se genero de UNA SOLA distribucion:")
print("  Normal(3000, 800), sin relacion con region, segmento, canal ni nada mas.")
print()
print("Es decir: por construccion NO existe ningun efecto que encontrar.")
print("Cualquier diferencia que aparezca sera azar.")
print()
print(datos.head())

### Ejercicio 5.1 — Busca hasta encontrar

Recorre todas las combinaciones de las cinco variables de segmentación y, dentro de cada
subgrupo, prueba si su monto medio difiere del de todos los demás.

Cuenta cuántas pruebas hiciste y cuántas salieron «significativas».

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
segmentadoras = ["region", "segmento", "canal", "antiguo", "digital"]

resultados = []
for col in segmentadoras:
    for valor in datos[col].unique():
        mask   = datos[col] == valor
        dentro = datos.loc[mask, "monto"]
        fuera  = datos.loc[~mask, "monto"]
        p = stats.ttest_ind(dentro, fuera, equal_var=False).pvalue
        resultados.append({"variable": col, "valor": valor,
                           "n_subgrupo": int(mask.sum()), "p": p})

res = pd.DataFrame(resultados).sort_values("p").reset_index(drop=True)

print(f"Pruebas realizadas : {len(res)}")
print(f"Significativas     : {(res['p'] < 0.05).sum()}")
print()
print("Las cinco mas 'prometedoras':")
print(res.head(5).round(4).to_string(index=False))
print()
if (res["p"] < 0.05).any():
    top = res.iloc[0]
    print(f"Titular disponible: 'Los clientes de {top['variable']} = {top['valor']}")
    print(f"presentan montos significativamente distintos (p = {top['p']:.4f}).'")
    print()
    print("Y es completamente falso. No hay ningun efecto: lo generamos nosotros de")
    print("una sola distribucion. Ese p pequeno es azar, encontrado a fuerza de buscar.")
else:
    print("En esta corrida ninguna bajo de 0.05. Sube el numero de variables")
    print("segmentadoras o combina dos a la vez y aparecera: es cuestion de insistir.")

In [ ]:
# ── VERIFICACIÓN 5.1 ─────────────────────────────────────────────────────
r = [check_bool("hiciste 14 pruebas (4+3+3+2+2)", len(res) == 14,
                "una prueba por cada valor de cada variable segmentadora"),
     check_bool("el p más pequeño es notablemente menor que los demás",
                res["p"].min() < res["p"].median())]
print()
print(f"Con {len(res)} pruebas independientes al 5 %, la probabilidad de al menos")
print(f"un falso positivo es del {100*(1-0.95**len(res)):.1f} %. No es mala suerte: es aritmética.")
print()
print("5.1 OK" if all(r) else "Revisa 5.1")

### Ejercicio 5.2 — Y ahora corrige

Aplica Holm y FDR (Benjamini-Hochberg) a esos p-valores con
`statsmodels.stats.multitest.multipletests` y mira qué queda en pie.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
from statsmodels.stats.multitest import multipletests

res["sig_sin_corregir"] = res["p"] < 0.05
res["sig_holm"] = multipletests(res["p"], alpha=0.05, method="holm")[0]
res["sig_fdr"]  = multipletests(res["p"], alpha=0.05, method="fdr_bh")[0]

print(res[["variable", "valor", "n_subgrupo", "p",
           "sig_sin_corregir", "sig_holm", "sig_fdr"]].round(4).to_string(index=False))
print()
print("RESUMEN")
print(f"  Significativas sin corregir : {res['sig_sin_corregir'].sum()}")
print(f"  Significativas con Holm     : {res['sig_holm'].sum()}")
print(f"  Significativas con FDR      : {res['sig_fdr'].sum()}")
print()
print("Las correcciones hacen exactamente su trabajo: eliminan hallazgos que solo")
print("existian porque hicimos muchas pruebas. Y como en estos datos no hay ningun")
print("efecto real, eliminarlos TODOS es la respuesta correcta.")
print()
print("El umbral de Bonferroni, para comparar:")
print(f"  alpha corregido = 0.05 / {len(res)} = {0.05/len(res):.5f}")
print()
print("Lo que hay que llevarse: si tu analisis recorrio subgrupos, o corrige,")
print("o declara el resultado como EXPLORATORIO y busca confirmarlo en datos nuevos.")

In [ ]:
# ── VERIFICACIÓN 5.2 ─────────────────────────────────────────────────────
r = [check_bool("Holm no deja pasar más que sin corregir",
                res["sig_holm"].sum() <= res["sig_sin_corregir"].sum()),
     check_bool("FDR es intermedia o igual: no más estricta que Holm",
                res["sig_fdr"].sum() >= res["sig_holm"].sum()),
     check_bool("tras corregir no queda ningún hallazgo (correcto: no hay efecto real)",
                res["sig_holm"].sum() == 0,
                "si queda alguno, es un falso positivo que sobrevivió; comenta el caso en clase")]
print()
print("Bloque 5 COMPLETO — laboratorio terminado" if all(r) else
      "Revisa 5.2 (o discute el caso si sobrevivió algún hallazgo)")

---
# Cierre

### Checklist de salida

- [ ] Calculo t y p a mano y con `scipy`, y sé que dan lo mismo.
- [ ] Sé que el IC es el conjunto de μ₀ que no se rechazan, porque lo comprobé.
- [ ] Sé que si H₀ es verdadera el p-valor es uniforme entre 0 y 1.
- [ ] Medí una tasa de error tipo I y me salió 5 %.
- [ ] Medí una potencia y vi que con n pequeño era ridícula.
- [ ] Vi el mismo efecto volverse significativo solo por subir n.
- [ ] Hice p-hacking sobre datos sin efecto y encontré un «hallazgo».
- [ ] Sé aplicar Holm y FDR.

### Lo que quedó demostrado con números

| Bloque | Lo que viste |
|---|---|
| 1 | `ttest_1samp` no es magia: reproduce tu aritmética al sexto decimal. |
| 2 | El intervalo de confianza contiene el resultado de todas las pruebas posibles. |
| 3 | Bajo H₀ el p-valor es uniforme, y la potencia con n pequeño es del orden del 10 %. |
| 4 | Un efecto de d = 0.10 pasa de p = 0.59 a p < 0.00001 sin cambiar de tamaño. |
| 5 | 14 pruebas sobre datos sin ningún efecto producen un «hallazgo significativo». |

El bloque 3 y el bloque 5 son los que importan. El 3 muestra que el método cumple lo que
promete cuando lo usas como está diseñado. El 5 muestra lo fácil que es romperlo sin
darse cuenta.

### Reto para la próxima clase

Busca en tu trabajo un informe o un dashboard donde alguien haya declarado una diferencia
«significativa» y contesta tres cosas:

1. ¿Cuál era el tamaño del efecto en unidades del negocio?
2. ¿Cuántas comparaciones se hicieron en total?
3. ¿Se declaró la hipótesis antes o después de mirar los datos?

Si no puedes responder ninguna, ya sabes qué falta en ese informe.

### Clase 6

**Pruebas para dos grupos y para varios:** t de Welch, ANOVA y las alternativas no
paramétricas. Pasamos de «¿es distinto de este valor?» a «¿son distintos entre sí?»,
que es la pregunta que de verdad aparece en el trabajo.

---
*Estadística Descriptiva e Inferencial · Módulo 2 · Clase 5 · SEED = 42*